# Agent 第 17 课：Agent Memory & Session

Phase 1 第 5 课我们把 agent 记忆拆成**短期（对话上下文）** 和**长期（跨会话的事实 / 向量存储）**。这节看 Bedrock 托管 Agent 把这两层做成了什么，以及哪些情况你还得自己搓。

学完这节你能回答：
1. `sessionId` / session state / agent memory 三者是什么关系
2. Session state 能传什么、不能传什么
3. Agent memory 的适用场景（以及为什么它不是万能记忆）
4. 什么时候应该绕过托管、自己接 DynamoDB / Vector DB

## 1. 三层记忆一张表

| 层 | 对应 | 生命周期 | 存在哪 | 你改得了吗 |
|---|---|---|---|---|
| **对话上下文** | 一次 invoke_agent 内多轮 | 调用内 | 内存 | 自动 |
| **Session state** | 同一 sessionId 内多次 invoke | session TTL（默认 10 分钟，最长 24h） | AWS 内部 | 你可以读写 |
| **Agent memory** | 跨 sessionId（按 memoryId 串） | 最长 365 天 | AWS 托管 | 读、清除；写是 agent 自动总结 |
| **（你自己的）长期存储** | 上面都不够用时 | 永久 | Dynamo / Postgres / Vector DB | 完全自己控 |

**关键心智**：session 是"一次对话"，memory 是"同一个用户跨多次对话"。

In [ ]:
import boto3, os, json, uuid
os.environ.setdefault("AWS_REGION", "us-west-2")

agent   = boto3.client("bedrock-agent")
runtime = boto3.client("bedrock-agent-runtime")

# 假设你已经有一个 agent（第 13 课建的），这里放进去
AGENT_ID       = "XXXXXXXXXX"
AGENT_ALIAS_ID = "YYYYYYYYYY"

## 2. sessionId：最基本的连续对话

同一个 `sessionId` 多次调用 → AWS 把之前的 message history 保留，模型看到完整上下文。

TTL 到了（或你自己调 `end_session`）→ 这段 history 作废。

In [ ]:
def invoke(text, session_id):
    stream = runtime.invoke_agent(
        agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID,
        sessionId=session_id, inputText=text,
    )
    parts = []
    for ev in stream["completion"]:
        if "chunk" in ev:
            parts.append(ev["chunk"]["bytes"].decode())
    return "".join(parts)

sid = str(uuid.uuid4())
print(invoke("我叫 Siji，在做一个 agent 课程。", sid))
print(invoke("我刚才说我叫什么名字？", sid))   # 模型应该记得

## 3. Session state：在用户和 agent 之间传"结构化槽"

很多场景 message history 不够用 —— 你需要往 session 里塞**结构化数据**，让后续调用能读到。比如：
- 当前用户的 ID、时区、偏好
- 上一步工具返回的关键结果
- 文件指针（"用户刚传了 s3://.../doc.pdf"）

这就是 **sessionState**。

In [ ]:
# 传入：sessionState 给这一轮
resp_stream = runtime.invoke_agent(
    agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID,
    sessionId=sid,
    inputText="给我用我的时区显示现在几点。",
    sessionState={
        # 1) 会话属性：整个 session 都能读到
        "sessionAttributes": {
            "user_id": "u-123",
            "tz": "America/Los_Angeles",
        },
        # 2) prompt 属性：只在这一轮 prompt 里可用（适合传一次性上下文）
        "promptSessionAttributes": {
            "current_time_iso": "2026-04-20T15:30:00-07:00",
        },
        # 3) 也可以把上次 tool 的 invocationResult 直接填进来（Return Control 场景）
        # "invocationId": "...",
        # "returnControlInvocationResults": [...],
    },
)
for ev in resp_stream["completion"]:
    if "chunk" in ev:
        print(ev["chunk"]["bytes"].decode(), end="")

### 怎么用 session attributes

在 agent 的 **instruction** 或 Action Group 的 **Lambda** 里可以引用：

- Instruction 里用 `$session_attributes$` / `$prompt_session_attributes$` 占位符，AWS 会在 orchestration prompt 里自动替换成 JSON
- Lambda 收到的 event 里有 `sessionAttributes` 和 `promptSessionAttributes` 字段，直接读

**典型用法**：把 `user_id` 放进 sessionAttributes，Lambda 里用它去数据库查该用户的记录 —— 而不是让模型把 user_id 作为参数传，那样又慢又不安全。

## 4. Agent memory：跨 session 的长期记忆（托管版）

2024 之后 Bedrock Agent 加了**内置 memory**：

- 你给 agent 开启 memory（`memoryConfiguration`）
- 每次 session 结束时，**AWS 自动用一个 LLM 对这段对话做 summary**，写进 memory
- 下次同一个 `memoryId` 的新 session，agent 会把之前的 summary 作为上下文拿进来

**它实际干的事**：自动的"对话摘要 + 下次注入"。和 Phase 1 第 5 课里我们手写的 `summarize_and_save` 一回事。

打开方式：

In [ ]:
# 示例（伪代码）：update_agent 时加上 memoryConfiguration
#
# agent.update_agent(
#     agentId=AGENT_ID,
#     agentName="...",
#     agentResourceRoleArn="arn:aws:iam::...",
#     foundationModel="anthropic.claude-sonnet-4-6-20260101-v1:0",
#     instruction="...",
#     memoryConfiguration={
#         "enabledMemoryTypes": ["SESSION_SUMMARY"],
#         "storageDays": 30,    # 留 30 天
#     },
# )
# agent.prepare_agent(agentId=AGENT_ID)
print("(示例)")

### 调用时带 memoryId

同一个用户跨 session，就**传同一个 memoryId**（通常是你系统的 user_id）：

In [ ]:
# 第一次会话
sid1 = str(uuid.uuid4())
runtime.invoke_agent(
    agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID,
    sessionId=sid1, memoryId="user-siji",
    inputText="以后你称呼我 'boss'，我喜欢简短回答。",
)
# 结束 session（触发摘要写入 memory）
runtime.end_session(
    agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID, sessionId=sid1,
)

# 几天后 —— 新 session，同一个 memoryId
sid2 = str(uuid.uuid4())
# agent 在 orchestration 时会把 memory summary 作为上下文带进去
# 所以它应该记得要叫你 'boss'、要简短
print("(示例 flow)")

### 读 / 清除 memory

In [ ]:
# 读这个用户的 memory（能看到 AWS 总结出来的内容）
# r = runtime.get_agent_memory(
#     agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID,
#     memoryId="user-siji", memoryType="SESSION_SUMMARY",
# )
# print(r["memoryContents"])

# 清除（用户要求被遗忘 / 切租户时）
# runtime.delete_agent_memory(
#     agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID, memoryId="user-siji",
# )
print("(示例)")

## 5. 托管 memory 的边界：什么时候不够

Agent memory 是**摘要式**的（SESSION_SUMMARY），它**不**做：

- **精确回忆**：它记不住"用户 3 周前说他的 API key 是 X"—— 摘要会丢细节，且本身也不该留这种东西
- **向量检索**：没有语义搜索；想"查出所有和 billing 相关的历史对话"不行
- **结构化事实**：像 CRM 一样存 "用户的订阅计划 = Pro"，不合适 —— 模型写入时不够稳定
- **跨 agent 共享**：memory 是和 agent 绑的；另一个 agent 读不到

### 什么时候自己搓

| 需求 | 该用什么 |
|---|---|
| 结构化用户属性（plan、偏好、时区） | **Dynamo / Postgres** + 通过 `sessionAttributes` 注入 |
| 语义检索过往对话 | **Knowledge Base**（第 15 课）把历史对话当文档存进去 |
| 事件 / 动作历史（审计） | **CloudWatch / S3 结构化日志** |
| 跨 agent / 跨系统共享 | 自建 memory service，多个 agent 共读 |

**推荐实践**：
- **稳定事实**（user profile）放自己的 DB，每次调用通过 `sessionAttributes` 传进去
- **对话风格 / 偏好 / 上次聊到哪**：用 agent memory
- **历史语料的语义检索**：KB

三者组合才是完整记忆系统。

## 6. 一段实战模式：user profile 注入 + agent memory

伪代码：

In [ ]:
def invoke_for_user(user_id, text):
    # 1) 从你自己的 DB 取 user profile（快、稳、结构化）
    profile = {"plan": "pro", "tz": "America/Los_Angeles", "locale": "zh-CN"}

    # 2) 本次的 sessionId 可复用也可新建；memoryId 用 user_id（跨 session 长期记忆）
    sid = str(uuid.uuid4())

    stream = runtime.invoke_agent(
        agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID,
        sessionId=sid,
        memoryId=user_id,
        inputText=text,
        sessionState={
            "sessionAttributes": {
                "user_id":  user_id,
                "plan":     profile["plan"],
                "tz":       profile["tz"],
            }
        },
    )
    return "".join(ev["chunk"]["bytes"].decode() for ev in stream["completion"] if "chunk" in ev)

# invoke_for_user("u-123", "帮我总结一下我上周问过的问题。")
print("(pattern demo)")

## 7. 工程直觉

1. **sessionId ≠ 用户 ID**。sessionId 是一次对话的键，memoryId 才对应"同一用户"。搞混会导致用户互相看到对方的 memory —— 严重事故。
2. **敏感数据永远不要进 memory**。memory 是 LLM 摘要出来的，可能被诱导吐出；账号 / token / PII 放自己的 DB。
3. **sessionAttributes 是 agent 和业务系统之间最干净的耦合点**。把"谁、何时、什么权限"放这里，Lambda 直接读，比让模型来传可靠得多。
4. **`$prompt_session_attributes$` 占位符**是把业务上下文塞进 instruction 的推荐方式；很多人会错用成 hard-code，每次部署 agent 版本 —— 不必要。
5. **给用户"忘记我"的入口**。`delete_agent_memory` + 你自己 DB 里那份 profile 一起删。合规要求。
6. **MemoryId 粒度可以更细**：不一定只按 user；可以按 `user_id:project_id` 分开 —— 同一用户不同项目互不污染。

---

下一课：**观测（Observability）** —— 记忆解决了"agent 看得见历史"，观测解决了"你看得见 agent"。CloudWatch / X-Ray / invocation logging 怎么组合起来调 bug。